In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

In [27]:
#0. Reading the Dataset

df=pd.read_csv("The_Titanic_dataset.csv")

In [28]:
#0. Cleaning the Dataset and Fixing NaN Values

df.columns=df.iloc[0]
df=df.drop(index=0)
df=df.reset_index(drop=True)

df.columns.values[3]="name"

df=df.drop_duplicates()
df=df.drop_duplicates(subset=["name"])

df.loc[df["name"].str.contains("Mr."), "gender"] = "male"
df.loc[df["name"].str.contains("Mrs."), "gender"] = "female"
df.loc[df["name"].str.contains("Miss."), "gender"] = "female"
df.head()

,sn,pclass,survived,name,gender,age,family,fare,embarked,date
0,1,3,0,Mr. Anthony,male,42,0,7.55,NaN,01-Jan-90
2,2,3,0,Master. Eugene Joseph,male,?,2,20.25,S,02-Jan-90
3,3,2,0,"Abbott, Mr. Rossmore Edward",male,NaN,2,**,S,03-Jan-90
5,5,3,1,"Abelseth, Miss. Karen Marie",female,16,0,7.65,S,05-Jan-90
6,6,3,1,"Abelseth, Mr. Olaus Jorgensen",male,25,0,7.65,S,06-Jan-90


In [29]:
#0. Normalizing Data

df["age"]=df["age"].replace("?", None)
df["age"]=df["age"].astype(float)
df["age"]=df["age"].fillna(df["age"].mean())

df["gender"]=df["gender"].replace("?", None)
df["gender"]=df["gender"].astype(str).str.lower()
df=df[df["gender"].isin(["male", "female"])]
df["gender"]=df["gender"].map({"male": 0, "female": 1})

df["fare"]=df["fare"].replace("**", None)
df["fare"]=df["fare"].astype(float)

X=df[["pclass", "gender", "age", "fare"]]
y=df["survived"]

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)
X_train, X_test, y_train, y_test=train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [38]:
# 1. Instantiate the Model

X=X.apply(pd.to_numeric, errors='coerce')
X=X.fillna(X.mean(numeric_only=True))
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)
y_train=y_train.astype(int)
y_test=y_test.astype(int)

log_reg=LogisticRegression(max_iter=1000)

param_grid={
    "solver": ["liblinear", "lbfgs"],
    "C": [0.001, 0.01, 0.1, 1, 10, 100]
}
grid_log=GridSearchCV(log_reg, param_grid, cv=5, scoring='accuracy')
grid_log.fit(X_train, y_train)
best_log_model=grid_log.best_estimator_
print(grid_log.best_params_)

{'C': 1, 'solver': 'liblinear'}


2. I could not run penalty on my notebook due to an obstinate python error that prevents 'penalty' from being used due to it being outdated from the latest versions. However, after doing some of my own research, I conclude that L2 would have been the best penalty because the model performs better when coefficient sizes are reduced and not completely eliminated.

3. The best C value was 1 because that indicated a point that was more of an equilibrium between overfitting and underfitting. A C that is too large will lead a model to be very complex and highly prone to overfitting. Conversely, a C that is too small will cause a model to be very simple and be highly prone to underfitting. Therefore, a smaller but not too small value of C is the best.

In [40]:
#4. Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

y_pred=grid_log.predict(X_test)
y_prob=grid_log.predict_proba(X_test)[:,1]

accuracy=accuracy_score(y_test, y_pred)
precision=precision_score(y_test, y_pred)
recall=recall_score(y_test, y_pred)
AUC=roc_auc_score(y_test, y_prob)

print(f' Accuracy: {accuracy}')
print(f' Precision: {precision}')
print(f' Recall: {recall}')
print(f' AUC: {AUC}')

 Accuracy: 0.7576923076923077
 Precision: 0.6836734693877551
 Recall: 0.6767676767676768
 AUC: 0.8298513081121778


4. The most appropriate metric is recall because it helps provide information on the main goal: correctly indentifying all survivors and avoiding missing people. Since recall is the model's true positive (those who were predicted to survive and actually did) over the true positive plus false  negative (those who were predicted to have died but actuallly survived), it can indicate whether there are missing survivors, which is incredibly crucial to a person on the rescue team.

5. I think logistic regression is more appropriate than linear regression because the outcome whether one survived or died is categorial and not continuous. Since linear regression predicts continuous numerical values, it could produce invalid outputs such as proabilities of survival that are greater than 1 or less than 0.

6. Energy, valence, and speechiness are the best features because they relate directly to how upbeat and catchy a song is, which probably affects danceability the best. Also, these three features had the greatest positive correlation to danceability as shown on the heatmap.

In [44]:
#7. Lasso Regression

from sklearn.linear_model import Lasso

df2=pd.read_csv(r'C:\Users\jiazh\Downloads\top2019.csv')
X2=df2[["energy", "valence", "speechiness"]]
y2=df2["danceability"]

X2=X2.fillna(X2.mean())
y2=y2.fillna(y2.mean())
X_train2, X_test2, y_train2, y_test2=train_test_split(X2, y2, test_size=0.2, random_state=42)
lasso=Lasso()
param_grid={'alpha': [0.001, 0.001, 0.1, 1]}
grid_lasso=GridSearchCV(lasso, param_grid, cv=5)
grid_lasso.fit(X_train2, y_train2)

print(grid_lasso.best_params_)

{'alpha': 0.1}


7. Lasso regression implements feature selection that shrinks specific coefficients to zero. The alpha value control how strongly features are reduced: the better the alpha value, the stronger reduction. A higher alpha leads to fewer important features. So, a best parameter value of 0.1 indicates that there has been a moderate amount of regularization (a more-or-less balance between overfitting and underfitting).

In [49]:
#8. Ridge

from sklearn.linear_model import Ridge

ridge=Ridge()
param_grid={'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}
grid_ridge=GridSearchCV(ridge, param_grid, cv=5)
grid_ridge.fit(X_train2, y_train2)

print(grid_ridge.best_params_)

{'alpha': 10}


8. The alpha value in ridge works similarily to the one in lasso. However, the key difference is that in ridge, features are not removed, just heavily shrunk. Since the best parameter value is 10 here, while it is not bad when it comes to allowing the model to generalize better, it makes the model liable to underfit in its simplicity and miss patterns.